# Models for a product

A product still has `"name"`, `"price"`, and `"in_stock"`. This lesson checks those values, then adds an `id` when the route saves the product.

Run the cells from top to bottom. Each Try it is one step.


## A model

A **model** is the list of fields for one kind of thing. **Pydantic** is the tool FastAPI uses to check those fields. **BaseModel** is the model tool inside Pydantic.

A **class** is the block that names a model and its fields. The line starts with `class`, then the name, then a colon.


## The body the client sends

The **client** is the caller that sends the request. TestClient is a client in this notebook.

**ProductIn** is the model for that body. `str` means text. `float` means a number that can have a decimal point. `bool` means True or False.

On a model, those marks tell FastAPI what to check.


In [ ]:
from pydantic import BaseModel

class ProductIn(BaseModel):
    name: str
    price: float
    in_stock: bool

pen = ProductIn(name="Pen", price=1.5, in_stock=True)
print(pen.name)
print(pen.price)
print(pen.in_stock)


## A rule for the price

**Field** adds a rule to one field. `gt` means greater than. `Field(gt=0)` means the price must be greater than 0.


### Try it

Replace `price: float` with `price: float = Field(gt=0)`. The cell posts a price of `0`. After that change, it prints `422`. Before the change, it prints `200`.


In [2]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ProductIn(BaseModel):
    name: str
    price: float = Field(gt=0)
    in_stock: bool

app = FastAPI()

@app.post("/products")
def add_product(item: ProductIn):
    return item

with TestClient(app) as client:
    response = client.post(
        "/products",
        json={"name": "Pen", "price": 0, "in_stock": True},
    )
    print(response.status_code)


422


### Solution

Try the exercise above first. Unfold the next cell the same way as in lesson 1: click the blue bar on its left, then press Shift+Enter.


In [ ]:
# Solution
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ProductIn(BaseModel):
    name: str
    price: float = Field(gt=0)
    in_stock: bool

app = FastAPI()

@app.post("/products")
def add_product(item: ProductIn):
    return item

with TestClient(app) as client:
    response = client.post(
        "/products",
        json={"name": "Pen", "price": 0, "in_stock": True},
    )
    print(response.status_code)


## The value the route returns

**Product** is the model for the response. It has `name`, `price`, and `in_stock`, plus `id`. `int` means a whole number. The **server** is the app while it is running. The route on the server sets `id`. The client does not send `id`.

A real API uses two models so the body the client sends and the value the route returns can differ. ProductIn is the body. Product is the value that comes back, with the id the server assigns.


In [3]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ProductIn(BaseModel):
    name: str
    price: float = Field(gt=0)
    in_stock: bool

class Product(BaseModel):
    id: int
    name: str
    price: float = Field(gt=0)
    in_stock: bool

app = FastAPI()
saved = []

@app.post("/products")
def add_product(item: ProductIn):
    product = Product(
        id=len(saved) + 1,
        name=item.name,
        price=item.price,
        in_stock=item.in_stock,
    )
    saved.append(product)
    return product

with TestClient(app) as client:
    response = client.post(
        "/products",
        json={"name": "Pen", "price": 1.5, "in_stock": True},
    )
    print(response.json()["id"])
    print(len(saved))


1
1


### Try it

Replace `pass` with `saved.append(product)`. After that change, the cell prints `1`, then `1`. Before the change, it prints `1`, then `0`.


In [7]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ProductIn(BaseModel):
    name: str
    price: float = Field(gt=0)
    in_stock: bool

class Product(BaseModel):
    id: int
    name: str
    price: float = Field(gt=0)
    in_stock: bool

app = FastAPI()
saved = []

@app.post("/products")
def add_product(item: ProductIn):
    product = Product(
        id=len(saved) + 1,
        name=item.name,
        price=item.price,
        in_stock=item.in_stock,
    )
    saved.append(product)
    return product

with TestClient(app) as client:
    response = client.post(
        "/products",
        json={"name": "Pen", "price": 1.5, "in_stock": True},
    )
    print(response.json()["id"])
    print(len(saved))
    print(saved)


1
1
[Product(id=1, name='Pen', price=1.5, in_stock=True)]


### Solution

Try the exercise above first. Unfold the next cell the same way as in lesson 1: click the blue bar on its left, then press Shift+Enter.


In [ ]:
# Solution
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ProductIn(BaseModel):
    name: str
    price: float = Field(gt=0)
    in_stock: bool

class Product(BaseModel):
    id: int
    name: str
    price: float = Field(gt=0)
    in_stock: bool

app = FastAPI()
saved = []

@app.post("/products")
def add_product(item: ProductIn):
    product = Product(
        id=len(saved) + 1,
        name=item.name,
        price=item.price,
        in_stock=item.in_stock,
    )
    saved.append(product)
    return product

with TestClient(app) as client:
    response = client.post(
        "/products",
        json={"name": "Pen", "price": 1.5, "in_stock": True},
    )
    print(response.json()["id"])
    print(len(saved))


## A missing name

A **validation error** is the response FastAPI sends when the body does not match the model. Its status code is `422`.

This body has price and in_stock, and no name.


### Try it

Replace `pass` with `print(response.json()["detail"][0]["msg"])`. The cell already prints `422`. After that change, it also prints `Field required`.


In [16]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ProductIn(BaseModel):
    name: str
    price: float = Field(gt=0)
    in_stock: bool

class Product(BaseModel):
    id: int
    name: str
    price: float = Field(gt=0)
    in_stock: bool

app = FastAPI()

@app.post("/products")
def add_product(item: ProductIn):
    return item

with TestClient(app) as client:
    response = client.post(
        "/products",
        json={"price": 2.5, "in_stock": True},
    )
    print(response.status_code)
    print(response.json()["detail"][0]["msg"])


422
Field required


### Solution

Try the exercise above first. Unfold the next cell the same way as in lesson 1: click the blue bar on its left, then press Shift+Enter.


In [ ]:
# Solution
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ProductIn(BaseModel):
    name: str
    price: float = Field(gt=0)
    in_stock: bool

class Product(BaseModel):
    id: int
    name: str
    price: float = Field(gt=0)
    in_stock: bool

app = FastAPI()

@app.post("/products")
def add_product(item: ProductIn):
    return item

with TestClient(app) as client:
    response = client.post(
        "/products",
        json={"price": 2.5, "in_stock": True},
    )
    print(response.status_code)
    print(response.json()["detail"][0]["msg"])
